# Build a model directly

The same Swiss system as the four-group adapters, written with `add_entity`, `add_attribute` and `add_relation` instead of reading CSV, Excel, YAML or JSON.

| Plant | Class | Category | Capacity | Efficiency | Variable cost |
|---|---|---|---|---|---|
| `gen.CH_GAS` | `GenerationUnit` | `ccgt` | 400 MW | 0.58 | 45 MU/MWh |
| `gen.CH_NUC` | `GenerationUnit` | `nuclear` | 3000 MW | 0.33 | 9 MU/MWh |
| `gen.CH_WIND` | `WindGenerationUnit` | `wind` | 100 MW | — | 0 |
| `gen.CH_PV` | `SolarGenerationUnit` | `solar` | 6500 MW | — | 0 |
| `gen.CH_HYD_RES` | `HydroGenerationUnit` | `reservoir_hydro` | 8500 MW | 0.9 | 0 |
| `gen.CH_HYD_ROR` | `HydroGenerationUnit` | `run_of_river` | 4000 MW | 0.9 | 0 |

Demand is `demand.CH`, 58 TWh/year. The reservoir (`storage.CH_RES`, 8.8 TWh, 20 TWh/year inflow) and run-of-river (`storage.CH_ROR`, no stored energy, 16 TWh/year inflow) are separate basins. Wind and photovoltaics have no efficiency attribute.

Companion page: `docs/guides/build-a-model.md`.


## Setup

Find the repository and open an empty model with the schema and the two libraries the region link needs.


In [1]:
from pathlib import Path
import os
import sys

here = Path.cwd().resolve()
REPO = next(
    (p for p in (here, *here.parents) if (p / "cesdm" / "schemas" / "base").is_dir()),
    None,
)
if REPO is None:
    raise FileNotFoundError(
        "Could not find cesdm/schemas/base. Open this notebook from the cesdm-toolbox checkout."
    )

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from cesdm_toolbox import build_model_from_yaml

model = build_model_from_yaml(str(REPO / "cesdm" / "schemas" / "base"))
model.import_library(str(REPO / "cesdm" / "library" / "default_library"))
model.import_library(str(REPO / "cesdm" / "library" / "regions_library"))
CH = "region.country.CH"
print("Repository:", REPO)


Repository: /Users/demirayt/Downloads/cesdm-toolbox


## Demand

58 TWh/year is stored as 58,000,000 MWh/year. The asset belongs to Switzerland, not to a bus: this is the same zonal system the adapters exchange.


In [2]:
model.add_entity("DemandUnit", "demand.CH")
model.add_relation("demand.CH", "belongsToGeographicalRegion", CH)
model.add_attribute("demand.CH", "annual_energy_demand", 58_000_000, unit="MWh/year")
print(model.get_entity("demand.CH").get_attr_value("annual_energy_demand"), "MWh/year")


58000000.0 MWh/year


## Gas and nuclear

Both are a generic `GenerationUnit`. The category is this system's own string, `ccgt` or `nuclear`, together with the efficiency and the variable cost. It is not a link to a library technology.


In [3]:
model.add_entity("GenerationUnit", "gen.CH_GAS")
model.add_relation("gen.CH_GAS", "belongsToGeographicalRegion", CH)
model.add_attribute("gen.CH_GAS", "generator_technology_type", "ccgt")
model.add_attribute("gen.CH_GAS", "nominal_power_capacity", 400, unit="MW")
model.add_attribute("gen.CH_GAS", "energy_conversion_efficiency", 0.58, unit="fraction")
model.add_attribute("gen.CH_GAS", "variable_operating_cost", 45, unit="MU/MWh")

model.add_entity("GenerationUnit", "gen.CH_NUC")
model.add_relation("gen.CH_NUC", "belongsToGeographicalRegion", CH)
model.add_attribute("gen.CH_NUC", "generator_technology_type", "nuclear")
model.add_attribute("gen.CH_NUC", "nominal_power_capacity", 3000, unit="MW")
model.add_attribute("gen.CH_NUC", "energy_conversion_efficiency", 0.33, unit="fraction")
model.add_attribute("gen.CH_NUC", "variable_operating_cost", 9, unit="MU/MWh")

for eid in ("gen.CH_GAS", "gen.CH_NUC"):
    plant = model.get_entity(eid)
    print(
        eid,
        plant.get_attr_value("generator_technology_type"),
        plant.get_attr_value("nominal_power_capacity"), "MW",
        "eta", plant.get_attr_value("energy_conversion_efficiency"),
        "cost", plant.get_attr_value("variable_operating_cost"),
    )


gen.CH_GAS ccgt 400.0 MW eta 0.58 cost 45.0
gen.CH_NUC nuclear 3000.0 MW eta 0.33 cost 9.0


## Wind and photovoltaics

These two have their own classes. Their efficiency cell is empty in the tutorial files, so no `energy_conversion_efficiency` is written. An empty cell is not stored as zero.


In [4]:
model.add_entity("WindGenerationUnit", "gen.CH_WIND")
model.add_relation("gen.CH_WIND", "belongsToGeographicalRegion", CH)
model.add_attribute("gen.CH_WIND", "generator_technology_type", "wind")
model.add_attribute("gen.CH_WIND", "nominal_power_capacity", 100, unit="MW")
model.add_attribute("gen.CH_WIND", "variable_operating_cost", 0, unit="MU/MWh")

model.add_entity("SolarGenerationUnit", "gen.CH_PV")
model.add_relation("gen.CH_PV", "belongsToGeographicalRegion", CH)
model.add_attribute("gen.CH_PV", "generator_technology_type", "solar")
model.add_attribute("gen.CH_PV", "nominal_power_capacity", 6500, unit="MW")
model.add_attribute("gen.CH_PV", "variable_operating_cost", 0, unit="MU/MWh")

for eid in ("gen.CH_WIND", "gen.CH_PV"):
    plant = model.get_entity(eid)
    eta = plant.get_attr_value("energy_conversion_efficiency")
    print(eid, plant.get_attr_value("nominal_power_capacity"), "MW", "eta", eta if eta is not None else "—")


gen.CH_WIND 100.0 MW eta —
gen.CH_PV 6500.0 MW eta —


## Reservoir and run-of-river

Each hydro plant draws from its own basin. The reservoir stores 8.8 TWh and receives 20 TWh/year. Run-of-river stores nothing and receives 16 TWh/year. The link is `drawsFromHydroReservoir`, not a convention between two tables.


In [5]:
model.add_entity("HydroGenerationUnit", "gen.CH_HYD_RES")
model.add_relation("gen.CH_HYD_RES", "belongsToGeographicalRegion", CH)
model.add_attribute("gen.CH_HYD_RES", "generator_technology_type", "reservoir_hydro")
model.add_attribute("gen.CH_HYD_RES", "nominal_power_capacity", 8500, unit="MW")
model.add_attribute("gen.CH_HYD_RES", "energy_conversion_efficiency", 0.9, unit="fraction")
model.add_attribute("gen.CH_HYD_RES", "variable_operating_cost", 0, unit="MU/MWh")
model.add_entity("HydroReservoir", "storage.CH_RES")
model.add_attribute("storage.CH_RES", "energy_storage_capacity", 8_800_000, unit="MWh")
model.add_attribute("storage.CH_RES", "annual_natural_inflow_energy", 20_000_000, unit="MWh/year")
model.add_relation("gen.CH_HYD_RES", "drawsFromHydroReservoir", "storage.CH_RES")

model.add_entity("HydroGenerationUnit", "gen.CH_HYD_ROR")
model.add_relation("gen.CH_HYD_ROR", "belongsToGeographicalRegion", CH)
model.add_attribute("gen.CH_HYD_ROR", "generator_technology_type", "run_of_river")
model.add_attribute("gen.CH_HYD_ROR", "nominal_power_capacity", 4000, unit="MW")
model.add_attribute("gen.CH_HYD_ROR", "energy_conversion_efficiency", 0.9, unit="fraction")
model.add_attribute("gen.CH_HYD_ROR", "variable_operating_cost", 0, unit="MU/MWh")
model.add_entity("HydroReservoir", "storage.CH_ROR")
model.add_attribute("storage.CH_ROR", "energy_storage_capacity", 0, unit="MWh")
model.add_attribute("storage.CH_ROR", "annual_natural_inflow_energy", 16_000_000, unit="MWh/year")
model.add_relation("gen.CH_HYD_ROR", "drawsFromHydroReservoir", "storage.CH_ROR")

for eid, basin in (("gen.CH_HYD_RES", "storage.CH_RES"), ("gen.CH_HYD_ROR", "storage.CH_ROR")):
    store = model.get_entity(basin)
    print(
        eid, "→", model.get_entity(eid).get_relation("drawsFromHydroReservoir"),
        store.get_attr_value("energy_storage_capacity"), "MWh,",
        store.get_attr_value("annual_natural_inflow_energy"), "MWh/year",
    )


gen.CH_HYD_RES → storage.CH_RES 8800000.0 MWh, 20000000.0 MWh/year
gen.CH_HYD_ROR → storage.CH_ROR 0.0 MWh, 16000000.0 MWh/year


## Check it

`validate()` checks the schema. An empty list means the file is correct CESDM. `validate_for_analysis("optimal_dispatch")` asks for what an hourly dispatch still needs, including demand and inflow profiles. This model has the annual totals only, so that second list is not empty.


In [6]:
errors = model.validate()
print(f"validate(): {len(errors)} schema errors")
gaps = model.validate_for_analysis("optimal_dispatch")
print(f"optimal_dispatch: {len(gaps)} gaps")
for line in gaps:
    print(" ", line)


validate(): 0 schema errors
optimal_dispatch: 7 gaps
  [optimal_dispatch] [DemandUnit:demand.CH] missing required 'hasDemandProfile'
  [optimal_dispatch] [WindGenerationUnit:gen.CH_WIND] missing required 'hasAvailabilityProfile'
  [optimal_dispatch] [WindGenerationUnit:gen.CH_WIND] missing required 'annual_resource_potential'
  [optimal_dispatch] [SolarGenerationUnit:gen.CH_PV] missing required 'hasAvailabilityProfile'
  [optimal_dispatch] [SolarGenerationUnit:gen.CH_PV] missing required 'annual_resource_potential'
  [optimal_dispatch] [HydroReservoir:storage.CH_RES] missing required 'hasNaturalInflowProfile'
  [optimal_dispatch] [HydroReservoir:storage.CH_ROR] missing required 'hasNaturalInflowProfile'


## A unit the schema does not allow

`annual_energy_demand` is stored in MWh/year. Writing 58 with unit `TWh` is rejected when the attribute is set, before `validate()` runs. The demand already stored on `demand.CH` stays 58,000,000 MWh/year.


In [7]:
try:
    model.add_attribute("demand.CH", "annual_energy_demand", 58, unit="TWh")
except ValueError as exc:
    print("rejected:", exc)
print("demand still", model.get_entity("demand.CH").get_attr_value("annual_energy_demand"), "MWh/year")


rejected: [DemandUnit:demand.CH] Unit 'TWh' is not allowed for 'annual_energy_demand'. Allowed: ['MWh/year']
demand still 58000000.0 MWh/year


## Save it, and compare with Group A's adapter

The YAML is the exchange file. Group A's adapter reads the tutorial CSV and writes the same system. `compare` matches plants on region and category, then checks capacity, efficiency, variable cost, storage and inflow. Identifiers are not part of that comparison. Zero differences means this notebook describes the same system as `adapter_a.py`.


In [8]:
out = REPO / "output" / "build-a-model"
out.mkdir(parents=True, exist_ok=True)
model.export_yaml(out / "swiss.yaml")
print("wrote", (out / "swiss.yaml").relative_to(REPO))

groups = REPO / "docs" / "examples" / "four-research-groups"
if str(groups) not in sys.path:
    sys.path.insert(0, str(groups))

import adapter_a
from compare import compare

official = adapter_a.leeres_modell()
adapter_a.to_cesdm(groups / "group-a-csv", official)
diffs = compare(model, official, "notebook", "adapter_a")
print(f"{len(diffs)} differences")
for line in diffs:
    print(" ", line)


wrote output/build-a-model/swiss.yaml
0 differences
